### 1) Introducción y clonado

El modelo que se va a usar, se ha obtenido del Github AlexeyAB, donde está implementado (https://github.com/AlexeyAB/darknet). Para trabajar con él se clona el repositorio de la arquitectura Yolov4-tiny.

Se explica a continuación cómo funciona este modelo y sus características principales. Este modelo se basa en la arquitectura YOLO que fue propuesta originalmente por Joseph Redmon (https://arxiv.org/pdf/1506.02640), y posteriormente AlexeyAB hizo un estudio y implemento una versión mejor del modelo que es la que se ha utilizado en este proyecto (https://arxiv.org/pdf/2004.10934). La arquitectura original se basa en GoogleNet, en total tiene 137 capas convolucionales, cuatro capas de maxpooling y dos capas totalmente conectadas. En cambio, el modelo implementado tiene 29 capas convolucionales, con función de activación leaky ReLU, y no tiene capas totalmente conectadas. Por el contrario, tiene dos capas de salida yolo. Cada capa de salida procesa características a una resolución diferente, derivadas de una parte específica del mapa de características, para conseguir detectar objetos de diferentes tamaños con precisión.

Este modelo está entre los detectores de una etapa, que tienen una gran velocidad para dectectar objetos acompañado de una alta precisión. Esto es especialmente relevante si vamos a querer que el modelo funcione correctamente en pruebas de tiempo real. La arquitectura YOLO funciona mejor que otras arquitecturas de detección de objetos (https://www.hitechbpo.com/blog/top-object-detection-models.php#:~:text=When%20compared%20to%20other%20real,challenging%20lighting%20and%20weather%20conditions.) pero, a la vez, tiene varias implementaciones en código abierto lo cual facilita mucho trabajar con él.

In [ ]:
!git clone https://github.com/AlexeyAB/darknet

Se hace un mount para trabajar en google drive:

In [ ]:

%cd ..
from google.colab import drive
drive.mount('/content/gdrive')

# se crea un link simbolico para que el path  /content/gdrive/My\ Drive/ProyectoVC sea igual a /mydrive
!ln -s /content/gdrive/My\ Drive/ProyectoVC /mydrive

# vemos los archivos que se van a usar
!ls /mydrive/yolov4-tiny

### 2) Cambios en el Makefile

Se hacen los cambios pertinentes en el makefile de darknet para que funcione en el entorno.

In [ ]:
%cd /content/darknet/
!sed -i 's/OPENCV=0/OPENCV=1/' Makefile
!sed -i 's/GPU=0/GPU=1/' Makefile
!sed -i 's/CUDNN=0/CUDNN=1/' Makefile
!sed -i 's/CUDNN_HALF=0/CUDNN_HALF=1/' Makefile
!sed -i 's/LIBSO=0/LIBSO=1/' Makefile

### 3) Se ejecuta el comando para crear darknet


In [ ]:
# build darknet
!make

### 4) Preparar los archivos a usar

Se copia todos los archivos que se van a utilizar desde google drive a Colab. Se borra el cfg predeterminado ya que se va a usar un fichero propio.

In [ ]:

%cd data/
!find -maxdepth 2 -type f -exec rm -rf {} ;
%cd ..

%rm -rf cfg/
%mkdir cfg

#### 4(a) Copiar el dataset

El dataset usado para el entrenamiento y los test del modelo está compuesto por imágenes de signos de la lengua de signos americana (ASL). 1512 imágenes para el entrenamiento del modelo,  144 para la validación y 72 para el test. Las imágenes están acompañadas de ficheros de anotación con formato xml. En estos ficheros se indica la ubicación de los objetos en forma de recuadros delimitadores y son necesarios para entrenar el modelo. Las imágenes están divididas en 26 clases, cada una es una letra del abecedario americano. Este es el enlace: https://public.roboflow.com/object-detection/american-sign-language-letters/1


In [ ]:
!cp /mydrive/yolov4-tiny/obj.voc.zip ../

!unzip ../obj.voc.zip -d data/

#### 4(b) Copiar el fichero yolov4-tiny.cfg

In [ ]:
!cp /mydrive/yolov4-tiny/yolov4-tiny.cfg ./cfg

#### 4(c) Copiar los ficheros obj.names y obj.data

In [ ]:
!cp /mydrive/yolov4-tiny/obj.names ./data
!cp /mydrive/yolov4-tiny/obj.data  ./data

#### 4(d) Copiar el programa processPrimero.py

Con este programa se crean los ficheros de texto "/content/darknet/data/train.txt" y  "/content/darknet/data/test.txt" que almacenan los nombres completos de las imágenes que van a ser usados para entrenar y para los test.

In [ ]:
!cp /mydrive/yolov4-tiny/processPrimero.py ./

In [ ]:
!python processPrimero.py

### 4(e) Copiar el programa ponerEtiquetas.py

Con este programa se cambia el formato de los labels de las imagenes al formato Yolo que es el que necesitan para su gestión.

In [ ]:
!cp /mydrive/yolov4-tiny/ponerEtiquetas.py ./
!python ponerEtiquetas.py

Nos aseguramos que el número de archivos es el mismo que en el otro programa ejecutado.

In [ ]:
!python ponerEtiquetas.py | wc -l

List the contents of the data folder to check if the train.txt and test.txt files have been created

In [ ]:
!more data/test.txt

### 5) Descargar los pesos pre-entrenados
Para hacer el transfer learning se descargan los pesos ya entrenados de las 29 capas convolucionales que tiene el modelo YOLOv4-tiny. Sobre estos pesos haremos un entrenamiento con las imágenes de nuestro dataset para ajustar el modelo a nuestro problema específico.

In [ ]:
!wget https://github.com/AlexeyAB/darknet/releases/download/darknet_yolo_v4_pre/yolov4-tiny.conv.29

### 6) Entrenamiento



Para el entrenamiento el modelo YOLOv4-tiny utiliza una función de pérdida compuesta para optimizar la detección de objetos. Esta función combina tres aspectos:

Pérdida de coordenadas: Penaliza las diferencias entre las coordenadas predichas y las reales de las cajas delimitadoras que acompañan a las imágenes.
Pérdida de confianza: Optimiza la predicción de la probabilidad de que una caja contenga un objeto.
Pérdida de clasificación: Evalúa la precisión en la clasificación del objeto detectado en una de las 26 clases.

Para la función de optimización Darknet utiliza SGD (Stochastic Gradient Descent) como el optimizador por defecto. Se ajustan sus parámetros en los ficheros de control. Algunos de los parámetros que se ajustan son la
tasa de aprendizaje (que decae a intervalos que tambien se ajustan), el momento (para mejorar la estabilidad del gradiente durante el entrenamiento) o regularizar los pesos para evitar el sobreajuste.

In [ ]:
!./darknet detector train data/obj.data cfg/yolov4-tiny.cfg yolov4-tiny.conv.29 -dont_show -map

Tenemos los mejores pesos obtenidos y los últimos.

In [ ]:
!ls  /mydrive/yolov4-tiny/backup/

#### Para no tener que empezar de nuevo a entrenar por si la conexión falla


In [ ]:
!./darknet detector train data/obj.data cfg/yolov4-tiny.cfg /mydrive/yolov4-tiny/backup/yolov4-tiny_last.weights -dont_show -map

###7) Test

Cambiamos el fichero de configuración para poder hacer el test

In [ ]:
%cd cfg
!sed -i 's/batch=64/batch=1/' yolov4-tiny.cfg
!sed -i 's/subdivisions=16/subdivisions=1/' yolov4-tiny.cfg
%cd ..

In [ ]:
!./darknet detector map data/obj.data cfg/yolov4-tiny.cfg /mydrive/yolov4-tiny/backup/yolov4-tiny_best.weights


Para medir los resultados del test se utilizan varias métricas diferentes como mAP (precisión promedio), IoU (intersección sobre unión), Precision, Recall y F1-Score.

Según metricas como la mAP del 0.8678 o el Recall del  0.88 parece indicar que el modelo detecta la mayoría de los objetos y tiene un buen rendimiento en general. El IoU Promedio = 65.60% de lo cual se puede deducir que las cajas delimitadoras predichas están razonablemente cerca de las cajas reales.

La precisión es del 0.79, lo cual es relativamente baja. Si nos fijamos en los positivos o negativos totales vemos que los falsos positivos totales son relativamente grandes, TP (true positive)= 63, FP (false positive) = 17, FN (false negative)= 9. Destacan las dos clases con ap = 0.00% (E y L), es decir, el modelo no ha aprendido a detectar esas clases en concreto.

Asi pues, el modelo tiene un rendimiento sólido. Sin embargo, la precisión no es muy alta. No a termiando de discernir entre las clases. Habría que trabajar en intentar mejorar la detección de algunas clases para así aumentar la precisión.



#### Ejecutar el detector en una imagen


In [ ]:

def imShow(path):
  import cv2
  import matplotlib.pyplot as plt
  %matplotlib inline

  image = cv2.imread(path)
  height, width = image.shape[:2]
  resized_image = cv2.resize(image,(3*width, 3*height), interpolation = cv2.INTER_CUBIC)

  fig = plt.gcf()
  fig.set_size_inches(18, 10)
  plt.axis("off")
  plt.imshow(cv2.cvtColor(resized_image, cv2.COLOR_BGR2RGB))
  plt.show('')

In [ ]:
!./darknet detector test data/obj.data cfg/yolov4-tiny.cfg /mydrive/yolov4-tiny/backup/yolov4-tiny_best.weights /content/darknet/data/test/C17_jpg.rf.ceb81f8ae3c3673bd060ebe71848eca8.jpg -thresh 0.2

imShow('predictions.jpg')


#### Ejecutar el detector en la webcam


In [ ]:
#Run detector on images captured by webcam for your custom YOLOv4-tiny trained model
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode

def take_photo(filename='photo.jpg', quality=0.8):
  js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = 'Capture';
      div.appendChild(capture);
      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});
      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();
      // Resize the output to fit the video element.
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
      // Wait for Capture to be clicked.
      await new Promise((resolve) => capture.onclick = resolve);
      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')
  display(js)
  data = eval_js('takePhoto({})'.format(quality))
  binary = b64decode(data.split(',')[1])
  with open(filename, 'wb') as f:
    f.write(binary)
  return filename

from IPython.display import Image
try:
  filename = take_photo()
  print('Saved to {}'.format(filename))

  # Show the image which was just taken.
  display(Image(filename))
except Exception as err:
  # Errors will be thrown if the user does not have a webcam or if they do not
  # grant the page permission to access it.
  print(str(err))

!./darknet detector test data/obj.data cfg/yolov4-tiny.cfg /mydrive/yolov4-tiny/backup/yolov4-tiny_best.weights photo.jpg -thresh 0.5
imShow('predictions.jpg')

## Ejecutar el detector en un video

In [ ]:

!./darknet detector demo data/obj.data cfg/yolov4-tiny.cfg /mydrive/yolov4-tiny/backup/yolov4-tiny_best.weights -dont_show /mydrive/yolo_test/ASL_test.mp4 -i 0.3 -out_filename /mydrive/yolo_test/result.avi

## Ejecutar el detector en la webcam en tiempo real

In [ ]:
# COdigo sacado de theAIGuysCode Github (https://github.com/theAIGuysCode/YOLOv4-Cloud-Tutorial/blob/master/yolov4_webcam.ipynb)


# import dependencies
from IPython.display import display, Javascript, Image
from google.colab.output import eval_js
from google.colab.patches import cv2_imshow
from base64 import b64decode, b64encode
import cv2
import numpy as np
import PIL
import io
import html
import time
import matplotlib.pyplot as plt
%matplotlib inline

# import darknet functions to perform object detections
from darknet import *

# load in our YOLOv4 architecture network
network, class_names, class_colors = load_network("cfg/yolov4-tiny.cfg", "data/obj.data", "/mydrive/yolov4-tiny/backup/yolov4-tiny_best.weights")
width = network_width(network)
height = network_height(network)

# darknet helper function to run detection on image
def darknet_helper(img, width, height):
  darknet_image = make_image(width, height, 3)
  img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
  img_resized = cv2.resize(img_rgb, (width, height),
                              interpolation=cv2.INTER_LINEAR)

  # get image ratios to convert bounding boxes to proper size
  img_height, img_width, _ = img.shape
  width_ratio = img_width/width
  height_ratio = img_height/height

  # run model on darknet style image to get detections
  copy_image_from_bytes(darknet_image, img_resized.tobytes())
  detections = detect_image(network, class_names, darknet_image)
  free_image(darknet_image)
  return detections, width_ratio, height_ratio

# function to convert the JavaScript object into an OpenCV image
def js_to_image(js_reply):
  """
  Params:
          js_reply: JavaScript object containing image from webcam
  Returns:
          img: OpenCV BGR image
  """
  # decode base64 image
  image_bytes = b64decode(js_reply.split(',')[1])
  # convert bytes to numpy array
  jpg_as_np = np.frombuffer(image_bytes, dtype=np.uint8)
  # decode numpy array into OpenCV BGR image
  img = cv2.imdecode(jpg_as_np, flags=1)

  return img

# function to convert OpenCV Rectangle bounding box image into base64 byte string to be overlayed on video stream
def bbox_to_bytes(bbox_array):
  """
  Params:
          bbox_array: Numpy array (pixels) containing rectangle to overlay on video stream.
  Returns:
        bytes: Base64 image byte string
  """
  # convert array into PIL image
  bbox_PIL = PIL.Image.fromarray(bbox_array, 'RGBA')
  iobuf = io.BytesIO()
  # format bbox into png for return
  bbox_PIL.save(iobuf, format='png')
  # format return string
  bbox_bytes = 'data:image/png;base64,{}'.format((str(b64encode(iobuf.getvalue()), 'utf-8')))

  return bbox_bytes

# JavaScript to properly create our live video stream using our webcam as input
def video_stream():
  js = Javascript('''
    var video;
    var div = null;
    var stream;
    var captureCanvas;
    var imgElement;
    var labelElement;

    var pendingResolve = null;
    var shutdown = false;

    function removeDom() {
       stream.getVideoTracks()[0].stop();
       video.remove();
       div.remove();
       video = null;
       div = null;
       stream = null;
       imgElement = null;
       captureCanvas = null;
       labelElement = null;
    }

    function onAnimationFrame() {
      if (!shutdown) {
        window.requestAnimationFrame(onAnimationFrame);
      }
      if (pendingResolve) {
        var result = "";
        if (!shutdown) {
          captureCanvas.getContext('2d').drawImage(video, 0, 0, 640, 480);
          result = captureCanvas.toDataURL('image/jpeg', 0.8)
        }
        var lp = pendingResolve;
        pendingResolve = null;
        lp(result);
      }
    }

    async function createDom() {
      if (div !== null) {
        return stream;
      }

      div = document.createElement('div');
      div.style.border = '2px solid black';
      div.style.padding = '3px';
      div.style.width = '100%';
      div.style.maxWidth = '600px';
      document.body.appendChild(div);

      const modelOut = document.createElement('div');
      modelOut.innerHTML = "<span>Status:</span>";
      labelElement = document.createElement('span');
      labelElement.innerText = 'No data';
      labelElement.style.fontWeight = 'bold';
      modelOut.appendChild(labelElement);
      div.appendChild(modelOut);

      video = document.createElement('video');
      video.style.display = 'block';
      video.width = div.clientWidth - 6;
      video.setAttribute('playsinline', '');
      video.onclick = () => { shutdown = true; };
      stream = await navigator.mediaDevices.getUserMedia(
          {video: { facingMode: "environment"}});
      div.appendChild(video);

      imgElement = document.createElement('img');
      imgElement.style.position = 'absolute';
      imgElement.style.zIndex = 1;
      imgElement.onclick = () => { shutdown = true; };
      div.appendChild(imgElement);

      const instruction = document.createElement('div');
      instruction.innerHTML =
          '<span style="color: red; font-weight: bold;">' +
          'When finished, click here or on the video to stop this demo</span>';
      div.appendChild(instruction);
      instruction.onclick = () => { shutdown = true; };

      video.srcObject = stream;
      await video.play();

      captureCanvas = document.createElement('canvas');
      captureCanvas.width = 640; //video.videoWidth;
      captureCanvas.height = 480; //video.videoHeight;
      window.requestAnimationFrame(onAnimationFrame);

      return stream;
    }
    async function stream_frame(label, imgData) {
      if (shutdown) {
        removeDom();
        shutdown = false;
        return '';
      }

      var preCreate = Date.now();
      stream = await createDom();

      var preShow = Date.now();
      if (label != "") {
        labelElement.innerHTML = label;
      }

      if (imgData != "") {
        var videoRect = video.getClientRects()[0];
        imgElement.style.top = videoRect.top + "px";
        imgElement.style.left = videoRect.left + "px";
        imgElement.style.width = videoRect.width + "px";
        imgElement.style.height = videoRect.height + "px";
        imgElement.src = imgData;
      }

      var preCapture = Date.now();
      var result = await new Promise(function(resolve, reject) {
        pendingResolve = resolve;
      });
      shutdown = false;

      return {'create': preShow - preCreate,
              'show': preCapture - preShow,
              'capture': Date.now() - preCapture,
              'img': result};
    }
    ''')

  display(js)

def video_frame(label, bbox):
  data = eval_js('stream_frame("{}", "{}")'.format(label, bbox))
  return data

# start streaming video from webcam
video_stream()
# label for video
label_html = 'Capturing...'
# initialze bounding box to empty
bbox = ''
count = 0
while True:
    js_reply = video_frame(label_html, bbox)
    if not js_reply:
        break

    # convert JS response to OpenCV Image
    frame = js_to_image(js_reply["img"])

    # create transparent overlay for bounding box
    bbox_array = np.zeros([480,640,4], dtype=np.uint8)

    # call our darknet helper on video frame
    detections, width_ratio, height_ratio = darknet_helper(frame, width, height)

    # loop through detections and draw them on transparent overlay image
    for label, confidence, bbox in detections:
      left, top, right, bottom = bbox2points(bbox)
      left, top, right, bottom = int(left * width_ratio), int(top * height_ratio), int(right * width_ratio), int(bottom * height_ratio)
      bbox_array = cv2.rectangle(bbox_array, (left, top), (right, bottom), class_colors[label], 2)
      bbox_array = cv2.putText(bbox_array, "{} [{:.2f}]".format(label, float(confidence)),
                        (left, top - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5,
                        class_colors[label], 2)

    bbox_array[:,:,3] = (bbox_array.max(axis = 2) > 0 ).astype(int) * 255
    # convert overlay of bbox into bytes
    bbox_bytes = bbox_to_bytes(bbox_array)
    # update bbox so next frame gets new overlay
    bbox = bbox_bytes